# Talks markdown generator for academicpages

Takes a TSV of talks with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `talks.py`. Run either from the `markdown_generator` folder after replacing `talks.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases, rather than Stuart's non-standard TSV format and citation style.

In [1]:
import pandas as pd
import os

## Data format

The TSV needs to have the following columns: title, type, url_slug, venue, date, location, talk_url, description, with a header at the top. Many of these fields can be blank, but the columns must be in the TSV.

- Fields that cannot be blank: `title`, `url_slug`, `date`. All else can be blank. `type` defaults to "Talk" 
- `date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. 
    - The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/talks/YYYY-MM-DD-[url_slug]`
    - The combination of `url_slug` and `date` must be unique, as it will be the basis for your filenames

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [2]:
!cat talks.tsv

title	highlighted	title_short	type	conference	conf_short	conf_type	conf_url	venue	date	location	slides	abstract	notes
Una forma 'barata' de garantizar triángulos en una gráfica (A 'cheap' way of guarantee triangles in a graph)		trianglesProb	Contributed talk	"26th Colloquium ""Victor Neumann-Lara"" of graph theory, combinatorics and its applications"	vnl2011	National		Universidad Autónoma del Estado de Hidalgo	06.03.11	Pachuca, Hgo. Mexico	yes		
Permutaciones: ¿Funciones biyectivas o simples revolturas? (Permutations: bijections or just jumbles?)		permutations	Contributed talk	44th National Conference of the Mexican Mathematical Society	smm2011	National		Universidad Autónoma de San Luis Potosí	12.10.11	San Luis Potosí, SLP. Mexico	yes	Dado un conjunto $A$ con $n$ elementos, las permutaciones de los elementos de dicho conjunto tienen dos grandes formas de estudio: Una de ellas es como objetos de carácter combinatorio, muy útiles para resolver variados problemas de conteo relacionados co

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [8]:
talks = pd.read_csv("talks.tsv", sep="\t", header=0)
talks

,title,highlighted,title_short,type,conference,conf_short,conf_type,conf_url,venue,date,location,slides,abstract,notes
0,Una forma 'barata' de garantizar triángulos en...,NaN,trianglesProb,Contributed talk,"26th Colloquium ""Victor Neumann-Lara"" of graph...",vnl2011,National,NaN,Universidad Autónoma del Estado de Hidalgo,06.03.11,"Pachuca, Hgo. Mexico",yes,NaN,NaN
1,Permutaciones: ¿Funciones biyectivas o simples...,NaN,permutations,Contributed talk,44th National Conference of the Mexican Mathem...,smm2011,National,NaN,Universidad Autónoma de San Luis Potosí,12.10.11,"San Luis Potosí, SLP. Mexico",yes,"Dado un conjunto $A$ con $n$ elementos, las pe...",NaN
2,Coloración en gráficas infinitas (Colorings o...,NaN,coloringsInfinite,Contributed talk,Discrete math seminar FisMat UMSNH,dmFisMat,Seminar,NaN,"Faculty of Physics and Mathematics, UMSNH",15.05.12,"Morelia, Mich. Mexico",yes,NaN,NaN
3,Poliedros regulares en pantalla de videojuegos...,NaN,regPolyhedra3Torus,Contributed talk,Discrete math seminar FisMat UMSNH,dmFisMat,Seminar,NaN,"Faculty of Physics and Mathematics, UMSNH",10.11.12,"Morelia, Mich. Mexico",yes,NaN,NaN
4,Poliedros regulares en el $3$-toro (Regular po...,NaN,regPolyhedra3Torus,Contributed talk,"28th Colloquium ""Victor Neumann-Lara"" of graph...",vnl2013,National,NaN,"Palacio Clavijero, Morelia",05.03.13,"Morelia, Mich. Mexico",yes,NaN,NaN
5,Poliedros regulares en el $3$-toro (Regular po...,NaN,regPolyhedra3Torus,Contributed talk,46th National Conference of the Mexican Mathem...,smm2013,National,NaN,Universidad Autónoma de Yucatán,28.10.13,"Mérida, Yuc. Mexico",yes,En los 70's Grünbaum publicó una lista de 47 p...,NaN
6,Poliedros regulares ¿Aún siguen de moda? (Reg...,NaN,regPolyhedra,Contributed talk,Graduate students seminar CCM,becariosCCM,Seminar,NaN,"Centro de Ciencias Matemáticas, UNAM Morelia",08.11.13,"Morelia, Mich. Mexico",no,NaN,NaN
7,Poliedros regulares en el $3$-toro (Regular po...,NaN,regPolyhedra3Torus,Talk by invitation,Graduate students seminar IM UNAM,becariosIM,Seminar,NaN,Institute of Mathematics UNAM,01.04.14,"Mexico City, Mexico",no,NaN,NaN
8,Regular polyhedra in the $3$-torus,yes,regPolyhedra3Torus,Contributed talk,5th Workshop Symmetries in Graphs Maps and Pol...,sigmap2014,International,http://mcs.open.ac.uk/SIGMAP/,ELIM Conference Centre,07.07.14,"West Malvern, U.K.",yes,Since Grünbaum's paper about regular polyhedra...,NaN
9,"Simetrías: un paseo por la geometría, los mosa...",NaN,mosaicos,Contributed talk,Degustaciones Matemáticas 2015,degustaciones2015,Local,NaN,"Faculty of Physics and Mathematics, UMSNH",14.02.15,"Morelia, Mich. Mexico",yes,NaN,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [9]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;",
    # "?": "&quest;",
    "¿": "&iquest;",
    "á": "&aacute;",
    "ä": "&auml;",
    "é": "&eacute;",
    "í": "&iacute;",
    "ó": "&oacute;",
    "ú": "&uacute;",
    "ü": "&uuml;",
    "Á": "&Aacute;",
    "É": "&Eacute;",
    "Í": "&Iacute;",
    "Ó": "&Oacute;",
    "Ú": "&Uacute;",
    "ñ": "&ntilde;",
    "Ñ": "&Ntilde;",
    }

def html_escape(text):
    if type(text) is str:
        return "".join(html_escape_table.get(c,c) for c in text)
    else:
        return "False"

def date_with_dashes(date):
    date_list=date.split(".")
    return f"20{date_list[2]}-{date_list[1]}-{date_list[0]}"

def date_as_list(date):
    return date.split(".")


In [40]:
date_with_dashes("20.01.23")

'2023-01-20'

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [14]:
loc_dict = {}

for row, item in talks.iterrows():
    
    html_filename = date_as_list(str(item.date))[2] + date_as_list(str(item.date))[1]+ "_" + item.conf_short + "_"+str(item.title_short)
    md_filename = html_filename + ".md"
    # year = item.date[:4]
    
    md = "---\ntitle: \""   + item.title + '"\n'
    md += "collection: talks" + "\n"
    
    if len(str(item.type)) > 3:
        md += 'type: "' + item.type + '"\n'
    else:
        md += 'type: "Other"\n'
    
    md += "permalink: /talks/" + html_filename + "\n"
    
    if len(str(item.venue)) > 3:
        md += 'venue: "' + item.venue + '"\n'
        
    if len(str(item.location)) > 3:
        md += "location: " + str(item.location) + "\n"
    
    if len(str(item.conference)) > 3:
        md += "conference: " + str(item.conference) + "\n"
        
    if str(item.slides) == "yes":
        md += "slides: " + "/slides/"+html_filename+".pdf\n"
    
    if len(str(item.conf_type)) >3:
        md += "conf_type: " + str(item.conf_type)+"\n"
    
    if str(item.highlighted) == "yes":
        md += "highlighted: yes\n"
    
    if len(str(item.notes)) >3:
        md += "notes: " + str(item.notes)+"\n"
    
    # if len(str(item.location)) > 3:
        md += 'date: "' + date_with_dashes(str(item.date)) + '"\n'
           
    md += "---\n"
    
    if len(str(item.conference)) > 3:
        md += "\n"+str(item.type)+" in the " + str(item.conference)+"\n" 
    
    if len(str(item.venue)) > 3:
        md += "\n"+str(item.venue)+"\n" 
    if len(str(item.location)) > 3:
        md += "\n"+str(item.location)+"\n" 

    if len(str(item.conf_url)) > 3:
        md += "\n[Conference website](" + item.conf_url + ")\n" 
    
    if str(item.slides) == "yes":
        md+="\n<i class=\"fa fa-file-pdf fa-fw\" aria-hidden=\"false\"></i>[Slides]({{ site.url }}/slides/"+html_filename+".pdf){:target='_blank'}\n"
    
    if len(str(item.abstract)) > 3:
        md += "\n**Abstract**: " + html_escape(item.abstract) + "\n"
        
        
    md_filename = os.path.basename(md_filename)
    #print(md)
    
    with open("../_talks/" + md_filename, 'w') as f:
        f.write(md)

These files are in the talks directory, one directory below where we're working from.

In [57]:
for row, item in talks.iterrows():
    if item.slides == "yes":
        print(date_as_list(str(item.date))[2] + date_as_list(str(item.date))[1]+ "_" + item.conf_short + "_"+str(item.title_short))

1103_vnl2011_trianglesProb
1110_smm2011_permutations
1205_dmFisMat_coloringsInfinite
1211_dmFisMat_regPolyhedra3Torus
1303_vnl2013_regPolyhedra3Torus
1310_smm2013_regPolyhedra3Torus
1407_sigmap2014_regPolyhedra3Torus
1502_degustaciones2015_mosaicos
1510_aquelarre15_toroids
1602_scdo2016_toroids
1606_esver2016_toroids
1608_enaeposmat2016_extensions
1610_aquelarre16_coversPolyhedra
1610_smm2017_twoToK
1702_cgtPeF_toroids
1707_esvec2017_twoToK
1708_prima2017_chiralExtensions
1711_becariosCimat_toroids
1803_vnl2018_extensionsToroids
1806_sigmap2018_extensionsToroids
1807_roglaPhDss_extensionsToroids
1808_enaeposmat2018_extensionsToroids
1810_smm2018_extensionsToroids
1906_9SiCGT_toroidalPolyhedra
1908_tomandoCiencia_whyPureMath
1912_cmswm2019_halving
2011_enm2020_toroids
2103_agtiw_extensions
2204_coloquioIM_extensions
2211_seminarKoper_voltageOperations
2306_10SiCGT_cayleyExtensions
2311_mathFisMat_regPolyhedra


In [7]:
!cat ../_talks/2013-03-01-tutorial-1.md

---
title: "Tutorial 1 on Relevant Topic in Your Field"
collection: talks
type: "Tutorial"
permalink: /talks/2013-03-01-tutorial-1
venue: "UC-Berkeley Institute for Testing Science"
date: 2013-03-01
location: "Berkeley CA, USA"
---

[More information here](http://exampleurl.com)

This is a description of your tutorial, note the different field in type. This is a markdown files that can be all markdown-ified like any other post. Yay markdown!
